# CardioIA – Fase 2, Parte 2: Classificador de Risco (baixo risco / alto risco)

Pipeline: TF-IDF → treino de `LogisticRegression` e `DecisionTreeClassifier` → comparação de métricas → escolha do melhor modelo → teste com frases novas.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_csv("../assets/base_risco.csv")

print("Total de frases:", len(df))
print(df["situacao"].value_counts())
df.head()

Total de frases: 60
situacao
alto risco     30
baixo risco    30
Name: count, dtype: int64


,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,tive uma dor forte no peito que irradia para o braço esquerdo,alto risco
2,estou com suor frio e aperto no peito há dez minutos,alto risco
3,sinto o coração disparado e quase desmaiei agora há pouco,alto risco
4,tenho falta de ar mesmo em repouso sem fazer nenhum esforço,alto risco


In [4]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    df["frase"], df["situacao"],
    test_size=0.2, stratify=df["situacao"], random_state=42,
)

print("Treino:", len(X_treino), "| Teste:", len(X_teste))
print(y_teste.value_counts())

Treino: 48 | Teste: 12
situacao
alto risco     6
baixo risco    6
Name: count, dtype: int64


In [5]:
vetorizador = TfidfVectorizer(lowercase=True)
X_treino_tfidf = vetorizador.fit_transform(X_treino)
X_teste_tfidf = vetorizador.transform(X_teste)

print("Vocabulário:", len(vetorizador.vocabulary_), "termos")
print("Shape treino:", X_treino_tfidf.shape)

Vocabulário: 210 termos
Shape treino: (48, 210)


In [6]:
modelo_lr = LogisticRegression(random_state=42)
modelo_lr.fit(X_treino_tfidf, y_treino)

modelo_dt = DecisionTreeClassifier(random_state=42)
modelo_dt.fit(X_treino_tfidf, y_treino)

print("Modelos treinados: Logistic Regression e Decision Tree")

Modelos treinados: Logistic Regression e Decision Tree


In [7]:
from sklearn.model_selection import cross_val_score
import numpy as np

print("🔄 AVALIAÇÃO DE ROBUSTEZ: Validação Cruzada (5 Folds)")
print("-" * 55)

# Aplica a validação cruzada no Logistic Regression
cv_scores_lr = cross_val_score(modelo_lr, X_treino_tfidf, y_treino, cv=5)
media_lr = np.mean(cv_scores_lr)
desvio_lr = np.std(cv_scores_lr)

print(f"Logistic Regression:")
print(f" -> Notas dos 5 testes: {cv_scores_lr}")
print(f" -> Acurácia Média: {media_lr * 100:.2f}% (+/- {desvio_lr * 100:.2f}%)\n")

# Aplica a validação cruzada na Decision Tree
cv_scores_dt = cross_val_score(modelo_dt, X_treino_tfidf, y_treino, cv=5)
media_dt = np.mean(cv_scores_dt)
desvio_dt = np.std(cv_scores_dt)

print(f"Decision Tree:")
print(f" -> Notas dos 5 testes: {cv_scores_dt}")
print(f" -> Acurácia Média: {media_dt * 100:.2f}% (+/- {desvio_dt * 100:.2f}%)")

🔄 AVALIAÇÃO DE ROBUSTEZ: Validação Cruzada (5 Folds)
-------------------------------------------------------
Logistic Regression:
 -> Notas dos 5 testes: [0.8        0.8        0.7        0.77777778 0.77777778]
 -> Acurácia Média: 77.11% (+/- 3.69%)

Decision Tree:
 -> Notas dos 5 testes: [0.5        0.9        0.8        0.77777778 0.77777778]
 -> Acurácia Média: 75.11% (+/- 13.34%)


In [6]:
pred_lr = modelo_lr.predict(X_teste_tfidf)
pred_dt = modelo_dt.predict(X_teste_tfidf)

acc_lr = accuracy_score(y_teste, pred_lr)
acc_dt = accuracy_score(y_teste, pred_dt)

print(f"Acurácia Logistic Regression: {acc_lr:.2f}")
print(f"Acurácia Decision Tree:       {acc_dt:.2f}")

print("\n=== Logistic Regression ===")
print(classification_report(y_teste, pred_lr))
print("Matriz de confusão:\n", confusion_matrix(y_teste, pred_lr, labels=["alto risco", "baixo risco"]))

print("\n=== Decision Tree ===")
print(classification_report(y_teste, pred_dt))
print("Matriz de confusão:\n", confusion_matrix(y_teste, pred_dt, labels=["alto risco", "baixo risco"]))

Acurácia Logistic Regression: 0.75
Acurácia Decision Tree:       0.92

=== Logistic Regression ===
              precision    recall  f1-score   support

  alto risco       0.80      0.67      0.73         6
 baixo risco       0.71      0.83      0.77         6

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighted avg       0.76      0.75      0.75        12

Matriz de confusão:
 [[4 2]
 [1 5]]

=== Decision Tree ===
              precision    recall  f1-score   support

  alto risco       0.86      1.00      0.92         6
 baixo risco       1.00      0.83      0.91         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

Matriz de confusão:
 [[6 0]
 [1 5]]


In [7]:
# Decision Tree teve maior acurácia neste split (48 treino / 12 teste) e foi o modelo escolhido.
modelo_final = modelo_dt

frases_novas = [
    "sinto uma dor no peito muito forte que não passa e estou suando frio",
    "tive uma leve dor de cabeça hoje, mas já melhorou depois de descansar",
    "acordei com falta de ar e o coração muito acelerado",
    "senti uma pontada rápida nas costas ao me abaixar para pegar algo",
]

X_novas_tfidf = vetorizador.transform(frases_novas)
predicoes = modelo_final.predict(X_novas_tfidf)

for frase, predicao in zip(frases_novas, predicoes):
    print(f"[{predicao}] {frase}")

[alto risco] sinto uma dor no peito muito forte que não passa e estou suando frio
[baixo risco] tive uma leve dor de cabeça hoje, mas já melhorou depois de descansar
[alto risco] acordei com falta de ar e o coração muito acelerado
[alto risco] senti uma pontada rápida nas costas ao me abaixar para pegar algo


# Reflexão Crítica: Padrões, Vieses e Governança Clínica

A experimentação com os modelos revelou comportamentos cruciais sobre a aplicação de Inteligência Artificial em ambientes de triagem médica:

## 1. A Ilusão da Acurácia e o Perigo do *Overfitting*
* **Desempenho inicial:** Embora a *Decision Tree* tenha atingido um pico de 92% de acurácia no teste inicial isolado (superando os 75% da *Logistic Regression*), a Validação Cruzada (*Cross-Validation*) revelou a verdadeira estabilidade dos modelos.
* **Variância:** A Árvore de Decisão apresentou alta variância, comprovando que sofreu de *overfitting* (memorizou os termos da base de treino em vez de aprender o padrão).
* **Conclusão clínica:** Em cenários clínicos com bases de dados restritas, modelos lineares como a *Logistic Regression* demonstram ser matematicamente mais conservadores e estáveis para generalização.

## 2. A Armadilha do TF-IDF e o Viés de Representatividade
* **Falha detectada:** No teste com dados inéditos, o modelo classificou a frase benigna *"senti uma pontada rápida nas costas ao me abaixar"* como alto risco.
* **Limitação do TF-IDF:** O método avalia a frequência das palavras, mas é cego para a semântica e o contexto.
* **Correlação espúria:** Como a base de treino continha pouquíssimos exemplos de dores osteomusculares rotuladas como baixo risco, o algoritmo associou as palavras *"pontada"* e *"costas"* a quadros graves (gerando um falso-positivo).

## 3. Impacto Clínico (Falsos Positivos vs. Falsos Negativos)
* **Peso dos erros:** Na triagem hospitalar real, os erros do modelo têm pesos diferentes:
  * **Falso-positivo:** Classificar dor muscular como infarto gera superlotação, ansiedade e desperdício de recursos.
  * **Falso-negativo:** Ignorar um infarto atípico pode ser fatal.
* **Falta de robustez:** O modelo atual, treinado com apenas 60 frases, não tem robustez para lidar com a complexidade e a ambiguidade dos relatos humanos.

## 4. Governança e Próximos Passos
Para que um sistema como este evolua de um protótipo acadêmico para uma ferramenta clínica viável, seria necessário:
* **Volume e Diversidade:** Expandir o *dataset* para dezenas de milhares de relatos reais, garantindo a representatividade de diferentes demografias e formas de expressão.
* **Evolução do Algoritmo:** Substituir o TF-IDF por modelos de processamento de linguagem natural que compreendem o contexto semântico (como *Word Embeddings* ou arquiteturas baseadas em *Transformers*, ex: BERT).
* **Human-in-the-Loop:** Garantir que a IA atue estritamente como sistema de apoio à decisão, mantendo a supervisão e o julgamento final a cargo dos profissionais de saúde.
